# Measurement Device (PMDCo): from data entry to RDF

This notebook shows how to register a measurement or characterization device
and convert that description into a standardised, machine-readable RDF graph
following the [OBI](http://purl.obolibrary.org/obo/obi.owl) /
[Platform MaterialDigital Core Ontology (PMDCo)](https://w3id.org/pmd/co/) conventions.

**You only need to edit one file:** `docs/example.input.json`. Everything
else is automatic.

A measurement device record in this schema captures:

- a **name** that uniquely identifies the physical unit in the lab
- the **manufacturer** and **model** for equipment traceability
- the **serial number** to distinguish individual units of the same model
- the **last calibration date** for quality assurance records

Once registered, the device's IRI can be referenced in experiment records
(e.g. `characterization/process/PMDCo/`) to link measurements to the instrument
that produced them.


## What the notebook does

```
example.input.json
  │  device name, manufacturer, model, serial number, calibration date
  │
  ▼
Transform
  │  converts your plain JSON into a structured OO-LD document
  │
  ▼
RDF graph
  │  machine-readable, ontology-aligned triples
  │
  ▼
SHACL validation  →  confirms the graph is structurally correct
SPARQL inspect    →  shows the key facts recorded in the graph
```


## Environment setup

If you are viewing this notebook on GitHub and want to run it locally, clone
the repository first so that all schema files and example inputs are present:

```bash
git clone https://github.com/Semantic-Dataspace/semantic-schemas.git
cd semantic-schemas
```

Then create a virtual environment and start Jupyter:

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install semantic-schemas jupyterlab
jupyter lab
```

Open this notebook from the `docs/` subfolder inside JupyterLab.

In [1]:
%pip install -q semantic-schemas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json, pathlib, rdflib
from semantic_schemas import Schema

HERE   = pathlib.Path().resolve()   # docs/
SCHEMA = HERE.parent                # measurement-device/PMDCo/

schema = Schema(SCHEMA)

print("Schema :", "/".join(SCHEMA.parts[-3:]))

Schema : schemas/measurement-device/PMDCo


## Step 1: Describe your device

Edit `docs/example.input.json` with your data, then run this cell to load it.

| Field | Required | Description |
|---|---|---|
| `device_name` | yes | A name that uniquely identifies this unit in the lab |
| `manufacturer` | no | Manufacturer name (e.g. `"Zwick Roell"`) |
| `model` | no | Model name or number (e.g. `"Z250"`) |
| `serial_number` | no | Manufacturer serial number |
| `calibration_date` | no | Last calibration date in ISO 8601 format (`YYYY-MM-DD`) |
| `description` | no | Free-text description of the device or its capabilities |
| `device_id` | no | Custom IRI slug; auto-derived from `device_name` if omitted |

**Why register a device?**  
Once saved to the knowledge graph, the device receives a stable IRI. That IRI
can be referenced in experiment records (`characterization/process/PMDCo/`) so
that every measurement is permanently linked to the instrument that produced it —
a key requirement for reproducibility and audit trails.

In [3]:
simplified = json.loads((HERE / "example.input.json").read_text())

print(json.dumps(simplified, indent=2))

{
  "device_name": "Zwick Z250 Universal Testing Machine #1",
  "manufacturer": "Zwick Roell",
  "model": "Z250",
  "serial_number": "ZR-12345",
  "calibration_date": "2024-03-01"
}


## Step 2: Convert to OO-LD

This step transforms your plain JSON into a structured
[OO-LD](https://github.com/OO-LD/oold-python) document.
Every field name is linked to a precise ontology term so machines can interpret
it unambiguously.

You can also run the transform from the command line:
```
python -m jsonata -e ../specs/transform.simplified.jsonata -i example.input.json
```

In [4]:
oold_doc = schema.transform(simplified)

print(json.dumps(oold_doc, indent=2))

{
  "conforms_to": "https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/measurement-device/PMDCo/#v1.0.0",
  "type": "obi:0000968",
  "id": "device-zwick-z250-universal-testing-machine-1",
  "label": "Zwick Z250 Universal Testing Machine #1",
  "manufacturer": "Zwick Roell",
  "model": "Z250",
  "serial_number": "ZR-12345",
  "calibration_date": "2024-03-01"
}


## Step 3: Convert to RDF

The OO-LD document is parsed into an RDF graph. The `@context` from
`specs/schema.oold.yaml` maps each field to its ontology IRI:

| JSON field | Ontology property |
|---|---|
| `type` | `rdf:type` → `obi:OBI_0000968` (Device) |
| `label` | `rdfs:label` |
| `manufacturer` | `schema:manufacturer` |
| `model` | `schema:model` |
| `serial_number` | `schema:serialNumber` |
| `calibration_date` | `dcterms:date` (typed `xsd:date`) |

In [5]:
flat = schema.to_graph(simplified)

print(f"Graph contains {len(flat)} triples.\n")
print(flat.serialize(format="turtle"))

Graph contains 7 triples.

@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix obo: <http://purl.obolibrary.org/obo/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix schema: <https://schema.org/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<https://w3id.org/pmd/co/test/device-zwick-z250-universal-testing-machine-1> a obo:OBI_0000968 ;
    rdfs:label "Zwick Z250 Universal Testing Machine #1" ;
    dcterms:conformsTo <https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/measurement-device/PMDCo/#v1.0.0> ;
    dcterms:date "2024-03-01"^^xsd:date ;
    schema:manufacturer "Zwick Roell" ;
    schema:model "Z250" ;
    schema:serialNumber "ZR-12345" .




In [6]:
# Optional: save to file
out_ttl = HERE / "output_device.ttl"
out_ttl.write_text(flat.serialize(format="turtle"))
print(f"Written to {out_ttl.name}")

Written to output_device.ttl


## Step 4: Validate against the SHACL shape

The shape file `specs/shape.ttl` checks that:

- The device has exactly one label.
- `dcterms:conformsTo` is present and is an IRI.
- `serial_number` (if given) is a plain string.
- `calibration_date` (if given) is an `xsd:date` value.

In [7]:
conforms, violations = schema.validate(flat)

print(f"Conforms: {conforms}")
for v in violations:
    print(f"  Violation: {v}")

Conforms: True


## Step 5: Inspect the graph

Confirm the key facts were recorded correctly.

In [8]:
OBI    = rdflib.Namespace("http://purl.obolibrary.org/obo/OBI_")
SCHEMA = rdflib.Namespace("https://schema.org/")
DCTERMS = rdflib.Namespace("http://purl.org/dc/terms/")

dev_iri = next(flat.subjects(rdflib.RDF.type, OBI["0000968"]))
label   = flat.value(dev_iri, rdflib.RDFS.label)
print(f"Device IRI : {dev_iri}")
print(f"Label      : {label}")

Device IRI : https://w3id.org/pmd/co/test/device-zwick-z250-universal-testing-machine-1
Label      : Zwick Z250 Universal Testing Machine #1


In [9]:
SPARQL_DEVICE = """
PREFIX obi:     <http://purl.obolibrary.org/obo/OBI_>
PREFIX rdfs:    <http://www.w3.org/2000/01/rdf-schema#>
PREFIX schema:  <https://schema.org/>
PREFIX dcterms: <http://purl.org/dc/terms/>

SELECT ?label ?manufacturer ?model ?serial ?calibrated
WHERE {
  ?dev a obi:0000968 ;
       rdfs:label ?label .
  OPTIONAL { ?dev schema:manufacturer ?manufacturer . }
  OPTIONAL { ?dev schema:model        ?model . }
  OPTIONAL { ?dev schema:serialNumber ?serial . }
  OPTIONAL { ?dev dcterms:date        ?calibrated . }
}
"""

rows = list(flat.query(SPARQL_DEVICE))
for r in rows:
    print(f"Name         : {r.label}")
    print(f"Manufacturer : {r.manufacturer or '—'}")
    print(f"Model        : {r.model or '—'}")
    print(f"Serial       : {r.serial or '—'}")
    print(f"Calibrated   : {r.calibrated or '—'}")

Name         : Zwick Z250 Universal Testing Machine #1
Manufacturer : Zwick Roell
Model        : Z250
Serial       : ZR-12345
Calibrated   : 2024-03-01


## Summary

| Step | What happens |
|---|---|
| 1 | You fill in `example.input.json` with the device name and metadata |
| 2 | The data is converted to an OO-LD document (ontology-mapped JSON) |
| 3 | The OO-LD is parsed into an RDF graph |
| 4 | The graph is validated against the SHACL shape |
| 5 | Key facts are extracted from the graph to confirm correctness |

The device's IRI (printed in Step 5) can now be used in:
- `characterization/process/PMDCo/` — to record which instrument was used in an experiment
- `expertise/` — under `measurement_devices` to record who is qualified to operate it


## Further reading

- [OO-LD primer](../../../docs/2_oold-primer.md): what OO-LD is and how it works
- [Schema format reference](../../../docs/3_schema-format.md): for schema authors
- [Characterization Process (PMDCo)](../../characterization/process/PMDCo/README.md): uses this device IRI
- [Expertise schema](../../expertise/README.md): links a person to the devices they operate
- [OBI Device class (OBI_0000968)](http://purl.obolibrary.org/obo/OBI_0000968)